# EMMa ASIN Splitting

In [ ]:
import json
from pathlib import Path
import numpy as np

from emma import EmmaDataset

In [ ]:
DATA_DIR = Path().resolve().parent.parent
DATA_CONFIG = DATA_DIR / "configs" / "data.json"
with DATA_CONFIG.open(encoding="utf-8") as f:
    data_config = json.load(f)

DATASET_ROOT = Path(data_config["emma_root"])
FILTERED_ASINS_PATH = Path(data_config["asins"]["post_augmentation"])

TRAIN_SPLIT = "Train"
VAL_SPLIT = "Val"
TEST_SPLIT = "Test"
MAX_TRAIN_ASINS = None
MAX_VAL_TRAIN_ASINS = 1000
MAX_VAL_ASINS = None
MAX_TEST_ASINS = None
SHUFFLE = False
RANDOM_SEED = 3621

TRAIN_ASINS_PATH = Path(data_config["asins"]["train"])
VAL_TRAIN_ASINS_PATH = Path(data_config["asins"]["val_train"])
VAL_ASINS_PATH = Path(data_config["asins"]["val"])
TEST_ASINS_PATH = Path(data_config["asins"]["test"])

In [ ]:
ds = EmmaDataset(DATASET_ROOT)

In [ ]:
filtered_asins = np.load(FILTERED_ASINS_PATH, allow_pickle=False).astype(str).reshape(-1)
filtered_asin_set = set(filtered_asins.tolist())

def build_split(split_name, max_items=None):
    if split_name not in ds.split_masks:
        raise ValueError(
            f"Unknown split {split_name!r}. Available splits: {list(ds.split_masks.keys())}"
        )

    split_mask = ds.split_masks[split_name].astype(bool, copy=False)
    split_asins = np.asarray(ds.asins[split_mask]).astype(str).reshape(-1)
    selected = np.asarray(
        [asin for asin in split_asins if asin in filtered_asin_set],
        dtype=str,
    )

    if SHUFFLE:
        rng = np.random.default_rng(RANDOM_SEED)
        rng.shuffle(selected)
    if max_items is not None:
        selected = selected[:max_items]
    if not len(selected):
        raise RuntimeError(f"No ASINs selected for split {split_name!r}.")

    return selected

In [ ]:
train_asins = build_split(TRAIN_SPLIT, MAX_TRAIN_ASINS)
val_asins = build_split(VAL_SPLIT, MAX_VAL_ASINS)
val_train_asins = build_split(VAL_SPLIT, MAX_VAL_TRAIN_ASINS)
test_asins = build_split(TEST_SPLIT, MAX_TEST_ASINS)

overlap = set(train_asins.tolist()) & set(val_asins.tolist()) & set(test_asins.tolist())
if overlap:
    raise RuntimeError(f"Train/validation overlap: {len(overlap)} ASINs")

print(f"{'Filtered':<24} ASINs: {len(filtered_asins):,}")
print(f"{'Train':<24} ASINs: {len(train_asins):,}")
print(f"{'Validation (training)':<24} ASINs: {len(val_train_asins):,}")
print(f"{'Validation':<24} ASINs: {len(val_asins):,}")
print(f"{'Test':<24} ASINs: {len(test_asins):,}")

In [ ]:
TRAIN_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
VAL_TRAIN_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
VAL_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
TEST_ASINS_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(TRAIN_ASINS_PATH, train_asins)
np.save(VAL_TRAIN_ASINS_PATH, val_train_asins)
np.save(VAL_ASINS_PATH, val_asins)
np.save(TEST_ASINS_PATH, test_asins)

print(f"Saved train ASINs to: {TRAIN_ASINS_PATH}")
print(f"Saved training-validation ASINs to: {VAL_TRAIN_ASINS_PATH}")
print(f"Saved validation ASINs to: {VAL_ASINS_PATH}")
print(f"Saved test ASINs to: {TEST_ASINS_PATH}")